## Model Interpretation & Error Analysis (Using Outputs from Notebook 04)

This notebook focuses on:
- selecting an operational threshold
- error analysis by time + geography
- model interpretation (what features drive predictions)
- report-ready summary tables

#### Load up stored data

In [0]:
from pyspark.sql import functions as SQL_FUNCTIONS
from pyspark.ml.functions import vector_to_array

FINAL_SELECTION_TABLE = "workspace.bda_taxi.model_comparison_final"
BEST_PRED_TABLE = "workspace.bda_taxi.model_preds_best"
THRESHOLD_TABLE = "workspace.bda_taxi.model_threshold_metrics"

final_selection = spark.table(FINAL_SELECTION_TABLE)
best_preds = spark.table(BEST_PRED_TABLE)

display(final_selection)
print("Best predictions rows:", best_preds.count())
display(best_preds.limit(5))

#### Threshold selection

Initially choosing the threshold that maximises F1 for the best model

In [0]:
threshold_metrics = spark.table(THRESHOLD_TABLE)
display(threshold_metrics.orderBy("threshold", "model_name"))

# Choose the threshold
best_model_name = final_selection.select("model").first()["model"]

best_threshold_row = (
    threshold_metrics
    .filter(SQL_FUNCTIONS.col("model_name") == best_model_name)
    .orderBy(SQL_FUNCTIONS.desc("f1_at_threshold"))
    .limit(1)
    .collect()[0]
)

CHOSEN_THRESHOLD = float(best_threshold_row["threshold"])
print("Chosen threshold:", CHOSEN_THRESHOLD, "for model:", best_model_name)
print("Row:", best_threshold_row)

In [0]:
scored = (
    best_preds
    .withColumn("prob_array", vector_to_array(SQL_FUNCTIONS.col("probability")))
    .withColumn("p_tip", SQL_FUNCTIONS.col("prob_array")[1].cast("double"))
    .withColumn("label_int", SQL_FUNCTIONS.col("label").cast("int"))
    .withColumn("pred_at_threshold", SQL_FUNCTIONS.when(SQL_FUNCTIONS.col("p_tip") >= CHOSEN_THRESHOLD, 1).otherwise(0))
    .withColumn("is_fp", SQL_FUNCTIONS.when((SQL_FUNCTIONS.col("pred_at_threshold") == 1) & (SQL_FUNCTIONS.col("label_int") == 0), 1).otherwise(0))
    .withColumn("is_fn", SQL_FUNCTIONS.when((SQL_FUNCTIONS.col("pred_at_threshold") == 0) & (SQL_FUNCTIONS.col("label_int") == 1), 1).otherwise(0))
    .withColumn("is_error", SQL_FUNCTIONS.when(SQL_FUNCTIONS.col("pred_at_threshold") != SQL_FUNCTIONS.col("label_int"), 1).otherwise(0))
)

scored.selectExpr(
    "count(*) as rows",
    "avg(is_error) as error_rate",
    "sum(is_fp) as false_positives",
    "sum(is_fn) as false_negatives"
).show(truncate=False)

#### Calibration check (probability vs actual tip rate)

In [0]:

# Create bins of predicted probability and compare predicted vs actual
calibration = (
    scored
    .withColumn("p_bin", (SQL_FUNCTIONS.floor(SQL_FUNCTIONS.col("p_tip") * 10) / 10).cast("double"))
    .groupBy("p_bin")
    .agg(
        SQL_FUNCTIONS.count("*").alias("trip_count"),
        SQL_FUNCTIONS.avg("p_tip").alias("avg_predicted_prob"),
        SQL_FUNCTIONS.avg(SQL_FUNCTIONS.col("label_int").cast("double")).alias("actual_tip_rate")
    )
    .orderBy("p_bin")
)

display(calibration)

CALIBRATION_TABLE = "workspace.bda_taxi.best_model_calibration_bins"
(
    calibration.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(CALIBRATION_TABLE)
)
print("Saved calibration table:", CALIBRATION_TABLE)

#### Store threshold selected

In [0]:
THRESHOLD_CHOICE_TABLE = "workspace.bda_taxi.best_model_threshold_choice"

threshold_choice_df = spark.createDataFrame(
    [(best_model_name, float(CHOSEN_THRESHOLD))],
    ["model_name", "chosen_threshold"]
).withColumn("chosen_ts", SQL_FUNCTIONS.current_timestamp())

(
    threshold_choice_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(THRESHOLD_CHOICE_TABLE)
)

print("Saved threshold choice:", THRESHOLD_CHOICE_TABLE)
display(spark.table(THRESHOLD_CHOICE_TABLE))

### Error analysis

#### Error analysis by time

In [0]:
# By time_bucket
if "time_bucket" in scored.columns:
    by_bucket = (
        scored.groupBy("time_bucket")
        .agg(
            SQL_FUNCTIONS.count("*").alias("trip_count"),
            SQL_FUNCTIONS.avg(SQL_FUNCTIONS.col("label_int").cast("double")).alias("actual_tip_rate"),
            SQL_FUNCTIONS.avg(SQL_FUNCTIONS.col("pred_at_threshold").cast("double")).alias("predicted_tip_rate"),
            SQL_FUNCTIONS.sum("is_fp").alias("false_positives"),
            SQL_FUNCTIONS.sum("is_fn").alias("false_negatives"),
            SQL_FUNCTIONS.avg("is_error").alias("error_rate")
        )
        .orderBy(SQL_FUNCTIONS.desc("trip_count"))
    )
    display(by_bucket)

# By pickup_hour
if "pickup_hour" in scored.columns:
    by_hour = (
        scored.groupBy("pickup_hour")
        .agg(
            SQL_FUNCTIONS.count("*").alias("trip_count"),
            SQL_FUNCTIONS.avg(SQL_FUNCTIONS.col("label_int").cast("double")).alias("actual_tip_rate"),
            SQL_FUNCTIONS.avg(SQL_FUNCTIONS.col("pred_at_threshold").cast("double")).alias("predicted_tip_rate"),
            SQL_FUNCTIONS.sum("is_fp").alias("false_positives"),
            SQL_FUNCTIONS.sum("is_fn").alias("false_negatives"),
            SQL_FUNCTIONS.avg("is_error").alias("error_rate")
        )
        .orderBy("pickup_hour")
    )
    display(by_hour)

#### Error analysis by geography

In [0]:
# Pickup zones
if "PU_Zone" in scored.columns:
    by_pu_zone = (
        scored.groupBy("PU_Zone")
        .agg(
            SQL_FUNCTIONS.count("*").alias("trip_count"),
            SQL_FUNCTIONS.avg(SQL_FUNCTIONS.col("label_int").cast("double")).alias("actual_tip_rate"),
            SQL_FUNCTIONS.avg(SQL_FUNCTIONS.col("pred_at_threshold").cast("double")).alias("predicted_tip_rate"),
            SQL_FUNCTIONS.sum("is_fp").alias("false_positives"),
            SQL_FUNCTIONS.sum("is_fn").alias("false_negatives"),
            SQL_FUNCTIONS.avg("is_error").alias("error_rate")
        )
        .orderBy(SQL_FUNCTIONS.desc("trip_count"))
    )
    display(by_pu_zone.limit(30))

# Dropoff zones
if "DO_Zone" in scored.columns:
    by_do_zone = (
        scored.groupBy("DO_Zone")
        .agg(
            SQL_FUNCTIONS.count("*").alias("trip_count"),
            SQL_FUNCTIONS.avg(SQL_FUNCTIONS.col("label_int").cast("double")).alias("actual_tip_rate"),
            SQL_FUNCTIONS.avg(SQL_FUNCTIONS.col("pred_at_threshold").cast("double")).alias("predicted_tip_rate"),
            SQL_FUNCTIONS.sum("is_fp").alias("false_positives"),
            SQL_FUNCTIONS.sum("is_fn").alias("false_negatives"),
            SQL_FUNCTIONS.avg("is_error").alias("error_rate")
        )
        .orderBy(SQL_FUNCTIONS.desc("trip_count"))
    )
    display(by_do_zone.limit(30))

#### False positive vs False negative concentration

Using a minimum number of trips set at 200 to ensure there is no impact from outliers (could see in notebook 03 in the visualisations the outliers)

In [0]:

def error_slices_by_column(scored_df, group_col: str, top_n: int = 25):
    return (
        scored_df.groupBy(group_col)
        .agg(
            SQL_FUNCTIONS.count("*").alias("trip_count"),
            SQL_FUNCTIONS.sum("is_fp").alias("false_positives"),
            SQL_FUNCTIONS.sum("is_fn").alias("false_negatives"),
            SQL_FUNCTIONS.avg("is_error").alias("error_rate")
        )
        .withColumn("fp_rate", SQL_FUNCTIONS.col("false_positives") / SQL_FUNCTIONS.col("trip_count"))
        .withColumn("fn_rate", SQL_FUNCTIONS.col("false_negatives") / SQL_FUNCTIONS.col("trip_count"))
        .orderBy(SQL_FUNCTIONS.desc("trip_count"))
    )

# Top FN rate zones (with minimum volume filter for fairness)
MIN_TRIPS = 200

if "PU_Zone" in scored.columns:
    pu_zone_slices = error_slices_by_column(scored, "PU_Zone")
    display(
        pu_zone_slices
        .filter(SQL_FUNCTIONS.col("trip_count") >= MIN_TRIPS)
        .orderBy(SQL_FUNCTIONS.desc("fn_rate"))
        .limit(25)
    )

if "DO_Zone" in scored.columns:
    do_zone_slices = error_slices_by_column(scored, "DO_Zone")
    display(
        do_zone_slices
        .filter(SQL_FUNCTIONS.col("trip_count") >= MIN_TRIPS)
        .orderBy(SQL_FUNCTIONS.desc("fn_rate"))
        .limit(25)
    )

#### Store data to be used for report

In [0]:
OUTPUT_SCORED_TABLE = "workspace.bda_taxi.best_model_scored"
OUTPUT_ERROR_BUCKET_TABLE = "workspace.bda_taxi.best_model_error_by_time_bucket"
ERROR_BY_HOUR_TABLE = "workspace.bda_taxi.best_model_error_by_pickup_hour"
ERROR_BY_PU_ZONE_TABLE = "workspace.bda_taxi.best_model_error_by_pu_zone"
ERROR_BY_DO_ZONE_TABLE = "workspace.bda_taxi.best_model_error_by_do_zone"

(
    scored.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(OUTPUT_SCORED_TABLE)
)
print("Saved:", OUTPUT_SCORED_TABLE)

if "time_bucket" in scored.columns:
    (
        by_bucket.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable(OUTPUT_ERROR_BUCKET_TABLE)
    )
    print("Saved:", OUTPUT_ERROR_BUCKET_TABLE)

if "pickup_hour" in scored.columns:
    (
        by_hour.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable(ERROR_BY_HOUR_TABLE)
    )
    print("Saved:", ERROR_BY_HOUR_TABLE)

if "PU_Zone" in scored.columns:
    (
        by_pu_zone.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable(ERROR_BY_PU_ZONE_TABLE)
    )
    print("Saved:", ERROR_BY_PU_ZONE_TABLE)

if "DO_Zone" in scored.columns:
    (
        by_do_zone.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable(ERROR_BY_DO_ZONE_TABLE)
    )
    print("Saved:", ERROR_BY_DO_ZONE_TABLE)

## Model interpretation (refitting best model only)

Refitting the best model that was stored in a table in previous notebook.
Depending on the model that it is:
- Logistic Regression: coefficients
- Random Forest: feature importances

##### Reuse the pipeline builders and feature lists from shared notebook 04

In [0]:
%run ./04_model_training_and_evaluation_shared



In [0]:
# Model interpretation (Random Forest) — feature importance by original columns
# This avoids one-hot feature name mapping and produces an interpretable importance per input column.

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier

TRAIN_TABLE = "workspace.bda_taxi.model_train_set"
TEST_TABLE = "workspace.bda_taxi.model_test_set"

train_df = spark.table(TRAIN_TABLE)
test_df = spark.table(TEST_TABLE)

label_column = "tipped"

# Use the same feature lists from the training notebook
numeric_cols = numeric_features
categorical_cols = categorical_features

# Index categoricals (no OneHot) to keep feature count small and interpretability high
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_cols
]

feature_cols = numeric_cols + [f"{c}_idx" for c in categorical_cols]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="keep"
)

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol=label_column,
    numTrees=30,
    maxDepth=8,
    subsamplingRate=0.7,
    featureSubsetStrategy="sqrt",
    # MUST be >= max categorical cardinality (zones ~259)
    maxBins=512,
    seed=42
)

pipeline = Pipeline(stages=indexers + [assembler, rf])
model = pipeline.fit(train_df)

rf_model = model.stages[-1]
importances = rf_model.featureImportances.toArray()

# Map importances back to original columns
importance_pairs = list(zip(feature_cols, [float(x) for x in importances]))
importance_pairs_sorted = sorted(importance_pairs, key=lambda x: x[1], reverse=True)

importance_df = spark.createDataFrame(importance_pairs_sorted, ["feature", "importance"])
display(importance_df.limit(30))